# 🧠 Ray English 자체 LLM — 무료로 내 출제 모델 만들기

**쓰는 법 (3가지만 하면 끝):**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장
2. 아래 **[2] 설정** 셀에서 `HF_USERNAME` 한 줄만 본인 HuggingFace 아이디로 바꾸기
3. 위 메뉴 **런타임 → 모두 실행** (또는 셀마다 왼쪽 ▶) → 마지막에 토큰 붙여넣기

> 학습 데이터(1,407문항)는 [4]에서 **자동으로 내려받습니다** — 아무것도 업로드하지 않아도 됩니다.  
> HuggingFace 쓰기 토큰: https://huggingface.co/settings/tokens (New token → **Write**)

In [ ]:
# %% [1] 설치 (Unsloth = 무료 T4에서 2배 빠르고 메모리 절약)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes huggingface_hub

### ⬇️ 여기 딱 한 줄만 바꾸세요
`HF_USERNAME` 을 본인 HuggingFace 아이디로 (예: `inheok1103`). 나머지는 그대로 두세요.

In [ ]:
# %% [2] 설정 — 여기만 취향껏 바꾸면 됨
MODEL_SIZE   = "3B"                    # "3B"(무료 T4 권장) 또는 "7B"(느리지만 더 똑똑, T4 빠듯)
HF_USERNAME  = "your-username"         # ← 본인 HuggingFace 아이디로 변경 (예: inheok1103)
MAX_SEQ_LEN  = 2048
EPOCHS       = 3                       # 샘플 ~1.4천 → 3 epoch 적정(적으면 학습부족, 많으면 과적합)
DATA_URL     = "https://raw.githubusercontent.com/inheok1103-alt/byeonghyeong-maker/master/selfllm/train.jsonl"

BASE_MODEL = {"3B": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
              "7B": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"}[MODEL_SIZE]
HF_REPO    = HF_USERNAME + "/ray-english-exam-" + MODEL_SIZE.lower()

In [ ]:
# %% [3] 모델 로드 (4bit) + LoRA 부착
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL, max_seq_length = MAX_SEQ_LEN,
    dtype = None, load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407,
)

In [ ]:
# %% [4] 데이터 자동 다운로드(우리 저장소) → Qwen 채팅 템플릿 적용
import os
if not os.path.exists("train.jsonl"):
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, "train.jsonl")   # 저장소에서 바로 받기 — 업로드 불필요
    print("train.jsonl 다운로드 완료")
from datasets import load_dataset
ds = load_dataset("json", data_files = "train.jsonl", split = "train")
def fmt(ex):
    return { "text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False) }
ds = ds.map(fmt)
print("학습 샘플:", len(ds))
print("--- 샘플 미리보기 ---")
print(ds[0]["text"][:500])

### ⏳ 아래가 실제 학습 — 30~60분 걸립니다
진행 로그(loss 숫자)가 계속 뜨면 정상입니다. 창을 닫지 마세요.

In [ ]:
# %% [5] 학습 (QLoRA) — assistant(정답) 부분만 학습하도록 마스킹
from trl import SFTTrainer
from transformers import TrainingArguments
try:
    from unsloth.chat_templates import train_on_responses_only
    _mask = True
except Exception:
    _mask = False
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = "text", max_seq_length = MAX_SEQ_LEN, packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 10, num_train_epochs = EPOCHS, learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10, optim = "adamw_8bit", weight_decay = 0.01,
        lr_scheduler_type = "linear", seed = 3407, output_dir = "outputs",
    ),
)
if _mask:  # 발문/지문(질문)은 손실에서 제외 → 정답 생성력만 강화
    try:
        trainer = train_on_responses_only(trainer,
            instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
    except Exception as e:
        print("마스킹 생략:", e)
trainer.train()

In [ ]:
# %% [6] 빠른 시험 — 우리 모델이 실제로 출제하는지 확인
FastLanguageModel.for_inference(model)
msgs = [
  {"role":"system","content":"너는 임용을 통과한 한국 고등학교 영어 내신 변형문제 출제 전문가다. 지문을 분석해 정확한 문항·워크북·분석을 만든다."},
  {"role":"user","content":"[유형] 글의순서\n[지문]\nWhen our actions clash with our beliefs, we feel an uncomfortable tension that the mind is eager to remove. Instead of admitting we were wrong, we quietly reshape our beliefs until they fit what we have already done.\n\n위 지문으로 '글의순서' 유형의 변형문항을 만들어라."},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.5, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

### 🔑 마지막 — 토큰 붙여넣기
실행하면 입력창이 뜹니다. 발급한 `hf_…` 토큰을 붙여넣고 Enter. (보안상 화면엔 안 보임 = 정상)

In [ ]:
# %% [7] 저장 — HuggingFace에 우리 모델 업로드(파일 소유) + GGUF(Ollama용)
from getpass import getpass
HF_TOKEN = getpass("HuggingFace 쓰기 토큰 붙여넣기(입력 숨김): ").strip()
# 병합 16bit 모델(파일 소유) push
model.push_to_hub_merged(HF_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
# Ollama에서 바로 쓰는 GGUF(q4_k_m)도 push
model.push_to_hub_gguf(HF_REPO + "-gguf", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
print("\n완료! 내 모델:", "https://huggingface.co/" + HF_REPO)
print("Ollama로 쓰기(고사양 PC/서버):  ollama run hf.co/" + HF_REPO + "-gguf")
print("→ 출제자 뇌 앱에서 무한모드가 이 로컬 모델(Ollama)을 1순위로 자동 사용")